In [0]:
spark.conf.set(
  "fs.azure.account.key.salesviewacc.dfs.core.windows.net",
  "<accesskey>"
)

In [0]:
df = spark.read.csv(
  "abfss://bronze@salesviewacc.dfs.core.windows.net/sales_view/product",
  header=True,
  inferSchema=True
)

print(df.columns)


In [0]:
import re

def to_snake_case(col):
    col = col.strip()
    col = re.sub(r'[^\w]+', '_', col)
    col = re.sub(r'(?<!^)(?=[A-Z])', '_', col).lower()
    return col

df = df.toDF(*[to_snake_case(c) for c in df.columns])

print(df.columns)


In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn("sub_category",
    when(col("category_id") == 1, "phone")
    .when(col("category_id") == 2, "laptop")
    .when(col("category_id") == 3, "playstation")
    .when(col("category_id") == 4, "e-device")
)


In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .save("abfss://silver@salesviewacc.dfs.core.windows.net/sales_view/product")


In [0]:
print(df.columns)